In [2]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../10. Near Miss 2016.xlsx"

xls=pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet="Near Miss"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("\nOriginal Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan",
    "Not Mentioned",
    "not mentioned",
    "NOT MENTIONED"
]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    empty=(
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    )

    if empty:
        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMNS
# ==========================

details_col=None
event_col=None

for col in df.columns:

    c=col.lower()

    if (
        "details" in c
        and "near" in c
    ):
        details_col=col

    if (
        "description" in c
        and "event" in c
    ):
        event_col=col

# ==========================
# REMOVE ROW ONLY IF BOTH EMPTY
# ==========================

if details_col and event_col:

    before=len(df)

    keep=(
        (
            df[details_col]
            .fillna("")
            .str.strip()
            !=""
        )
        |
        (
            df[event_col]
            .fillna("")
            .str.strip()
            !=""
        )
    )

    df=df[keep]

    print(
        "\nRows Removed:",
        before-len(df)
    )

# ==========================
# DATE CLEANING
# ==========================

required_dates=[

    "date_&_time_of_occurrence",
    "date_and_time_of_occurrence",
    "due_date",
    "closure_date_-_ca",
    "closure_date_ca"
]

date_cols=[]

for col in df.columns:

    c=col.lower()

    if c in required_dates:

        date_cols.append(col)


def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v=="":
        return np.nan

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned"
    ]:
        return np.nan


    # keep month only
    if re.match(
        r"^[A-Za-z]{3}$",
        v
    ):
        return v


    # keep already formatted dates
    # EX: 8-Nov-16
    if re.match(
        r"^\d{1,2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v


    # yyyy-mm-dd
    if re.match(
        r"^\d{4}-\d{2}-\d{2}",
        v
    ):

        try:

            d=pd.to_datetime(
                v,
                errors="coerce"
            )

            if pd.notna(d):

                return d.strftime(
                    "%d-%b-%y"
                )

        except:
            pass


    # numeric dates only
    if re.match(
        r"^\d{1,2}[/-]\d{1,2}[/-]\d{2,4}$",
        v
    ):

        try:

            d=pd.to_datetime(
                v,
                dayfirst=True,
                errors="coerce"
            )

            if pd.notna(d):

                return d.strftime(
                    "%d-%b-%y"
                )

        except:
            pass


    return v


for col in date_cols:

    df[col]=df[col].apply(
        clean_date
    )

print("\nFormatted Date Columns:")
print(date_cols)
# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# RESET SERIAL NUMBER
# ==========================

existing=[]

for col in df.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(":","")
        .replace(" ","")
    )

    if c in [
        "sino",
        "slno",
        "serialno"
    ]:
        existing.append(col)

if existing:

    df=df.drop(
        columns=existing,
        errors="ignore"
    )

df=df.reset_index(
    drop=True
)

df.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(df)+1
    )
)

print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_10_Near_Miss.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Sheets:
['Sheet1', 'Sheet2', 'Near Miss']

Original Shape: (486, 35)

Removed Empty Columns:
[]

Rows Removed: 1

Formatted Date Columns:
['Date_and_Time_of_occurrence', 'Due_Date', 'Closure_Date_-_CA']

Duplicates Removed: 0

SI_No reordered

Missing Summary:
Port_Name                                       336
Details_of_near_miss                              3
Description_of_event_leading_to_the_incident    134
Immediate_action_taken                            1
Potential_extent_of_damage/injury               146
Cause_Analysis                                   48
Proposed_Corrective_Action                       62
Details_of_potential_loss_Category               30
Immediate_action_initiated                       28
Learning_Potential                               38
Severity_of_incident                             36
Corrective_Action                                54
PIC                                             228
Due_Date                                        228
Status_of_C